# 🚀 R2AI2026 — Phương án A+B: Hybrid Search + BGE-Reranker
**Mục tiêu:** Tăng điểm từ 0.3976 lên 0.45-0.50 | **Không cần chạy lại LLM!**

### Tính năng chống mất dữ liệu:
- **Cell 6 (Retrieve):** Background thread tự động backup `retrieval.db` lên Drive **mỗi 5 phút**
- **Cell 7 (Rerank):** Auto-checkpoint mỗi 50 câu → Drive. Nếu bị ngắt, **chạy lại sẽ resume từ câu dở**

### 📁 File cần có trong Drive > `R2AI_Artifacts`:
- `sme_rag_clean.zip` (71KB) ✅
- `results_partial.jsonl` (6MB) ✅
- `R2AIStage1DATA.json` (518KB) ✅

## Cell 1 — GPU + Mount Drive + Kiểm tra tổng thể

In [ ]:
import subprocess, os
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('🖥️  GPU:', r.stdout.strip() or '❌ KHÔNG CÓ GPU!')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'\n📋 Scan {DRIVE_DIR}:')
for root, dirs, files in os.walk(DRIVE_DIR):
    level = root.replace(DRIVE_DIR, '').count(os.sep)
    indent = '  ' * level
    rel = root.replace(DRIVE_DIR, '') or '/'
    print(f'{indent}📁 {rel}/')
    for f in files:
        sz = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f'{indent}  📄 {f} ({sz:.1f}MB)')
print('\n✅ Cell 1 Done!')

## Cell 2 — Cài thư viện

In [ ]:
import subprocess, sys
print('📦 Cài thư viện...')
pkgs = [['datasets', 'huggingface_hub'], ['chromadb==0.5.23'],
        ['sentence-transformers>=2.7.0'], ['rank-bm25', 'numpy', 'tqdm']]
for pkg_list in pkgs:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkg_list
    print(f'  {" ".join(pkg_list)}...', end=' ')
    r = subprocess.run(cmd, capture_output=True, text=True)
    print('✅' if r.returncode == 0 else f'⚠️ {r.stderr[-80:]}')
print('\n✅ Cell 2 Done!')

## Cell 3 — Giải nén & Setup project

In [ ]:
import os, sys, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
WORK_DIR  = '/content/sme-legal-assistant'
REPO_URL  = 'https://github.com/Platypus27-coder/sme-legal-assistant.git'
BRANCH    = 'experiment-rag-upgrade'

if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)

print(f'⬇️ Clone branch {BRANCH} từ Github...')
BRANCH    = 'experiment-rag-upgrade'
if r.returncode != 0:
    print(f'❌ Lỗi Clone (Có thể do repo private cần Token): {r.stderr}')
    print('👉 Cách sửa: Thêm token vào URL -> https://<YOUR_TOKEN>@github.com/...')
    raise SystemExit

key_files = ['run.py', 'src/vpl/settings.py', 'rerank_retune.py', 'src/vpl/search/hybrid.py']
print('📋 Kiểm tra cấu trúc:')
for f in key_files:
    ok = os.path.exists(f'{WORK_DIR}/{f}')
    print(f'  {"✅" if ok else "❌"} {f}')

for d in ['output', 'cache', 'raw', 'index']:
    os.makedirs(f'{WORK_DIR}/artifacts/{d}', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

if os.path.exists(f'{DRIVE_DIR}/results_partial_test.jsonl'):
    shutil.copy2(f'{DRIVE_DIR}/results_partial_test.jsonl', f'{WORK_DIR}/artifacts/output/results_partial_test.jsonl')
    print('[OK] Restored results_partial_test.jsonl')
elif os.path.exists(f'{DRIVE_DIR}/results_partial.jsonl'):
if os.path.exists(f'{DRIVE_DIR}/results_partial_test.jsonl'):
    shutil.copy2(f'{DRIVE_DIR}/results_partial_test.jsonl', f'{WORK_DIR}/artifacts/output/results_partial_test.jsonl')
    print('[OK] Restored results_partial_test.jsonl')
elif os.path.exists(f'{DRIVE_DIR}/results_partial.jsonl'):
if os.path.exists(f'{DRIVE_DIR}/results_partial_test.jsonl'):
    shutil.copy2(f'{DRIVE_DIR}/results_partial_test.jsonl', f'{WORK_DIR}/artifacts/output/results_partial_test.jsonl')
    print('[OK] Restored results_partial_test.jsonl')
elif os.path.exists(f'{DRIVE_DIR}/results_partial.jsonl'):
    shutil.copy2(f'{DRIVE_DIR}/results_partial.jsonl', f'{WORK_DIR}/artifacts/output/results_partial_test.jsonl')
    print('[OK] Restored results_partial.jsonl as results_partial_test.jsonl')
    print('[OK] Restored results_partial.jsonl as results_partial_test.jsonl')
    print('[OK] Restored results_partial.jsonl as results_partial_test.jsonl')
shutil.copy2(f'{DRIVE_DIR}/R2AIStage1DATA.json',   f'{WORK_DIR}/data/R2AIStage1DATA.json')

sys.path.insert(0, f'{WORK_DIR}/src')
os.chdir(WORK_DIR)

from vpl.settings import SEARCH
print(f'\n✅ Import OK — high_conf={SEARCH.high_conf_threshold}, max_art={SEARCH.max_articles}')
print('✅ Cell 3 Done!')


## Cell 4 — Ingest (tận dụng Drive nếu có sẵn)

In [ ]:
import os, shutil, subprocess
DRIVE_DIR    = '/content/drive/MyDrive/R2AI_Artifacts'
CHUNKS_LOCAL = 'artifacts/raw/chunks.jsonl'

# Tìm chunks từ Drive (từ lần chạy trước)
candidates = [f'{DRIVE_DIR}/raw/chunks.jsonl', f'{DRIVE_DIR}/chunks.jsonl']
found = next((c for c in candidates if os.path.exists(c)), None)

if found:
    shutil.copy2(found, CHUNKS_LOCAL)
    n = sum(1 for _ in open(CHUNKS_LOCAL, encoding='utf-8'))
    print(f'✅ Chunks từ Drive: {n:,} chunks — Bỏ qua Ingest!')
else:
    print('🔄 Tải dữ liệu luật từ HuggingFace (~10-15 phút)...')
    result = subprocess.run(['python', 'run.py', 'ingest'], capture_output=False)
    if result.returncode == 0 and os.path.exists(CHUNKS_LOCAL):
        n = sum(1 for _ in open(CHUNKS_LOCAL, encoding='utf-8'))
        print(f'✅ {n:,} chunks')
        shutil.copy2(CHUNKS_LOCAL, f'{DRIVE_DIR}/chunks.jsonl')
        print('☁️  Backed up to Drive')
    else:
        print('❌ Ingest thất bại!')
print('\n✅ Cell 4 Done!')

## Cell 5 — Index: BM25 + ChromaDB

In [ ]:
import os, shutil, subprocess
DRIVE_DIR   = '/content/drive/MyDrive/R2AI_Artifacts'
BM25_LOCAL  = 'artifacts/index/bm25/corpus.pkl'
CHROMA_DB   = 'artifacts/index/chroma/chroma.sqlite3'
DRIVE_TAR   = f'{DRIVE_DIR}/index_built.tar.gz'
DRIVE_INDEX = f'{DRIVE_DIR}/index'

if os.path.exists(f'{DRIVE_INDEX}/bm25/corpus.pkl'):
    print('📦 Restore BM25 từ Drive...')
    os.makedirs('artifacts/index/bm25', exist_ok=True)
    for f in os.listdir(f'{DRIVE_INDEX}/bm25'):
        shutil.copy2(f'{DRIVE_INDEX}/bm25/{f}', f'artifacts/index/bm25/{f}')
    print(f'✅ BM25 OK | ChromaDB: {"✅" if os.path.exists(CHROMA_DB) else "❌ cần build mới (BGE-M3)"}')

if not os.path.exists(BM25_LOCAL) or not os.path.exists(CHROMA_DB):
    print('🔄 Build Index (BM25 + ChromaDB) — ~45 phút...')
    subprocess.run(['python', 'run.py', 'index', '--device', 'cuda', '--reset'], capture_output=False)

if os.path.exists(BM25_LOCAL):
    print('\n☁️  Backup index lên Drive...')
    subprocess.run(['tar', '-czf', DRIVE_TAR, '-C', 'artifacts', 'index'], capture_output=True)
    sz = os.path.getsize(DRIVE_TAR) / 1024 / 1024
    print(f'☁️  index_built.tar.gz ({sz:.0f}MB)')

for p, label in [(BM25_LOCAL,'BM25'), (CHROMA_DB,'ChromaDB')]:
    ok = os.path.exists(p)
    sz = f'{os.path.getsize(p)/1024/1024:.0f}MB' if ok else ''
    print(f'  {"✅" if ok else "❌"} {label} {sz}')
print('\n✅ Cell 5 Done!')

## Cell 6 — Hybrid Retrieve (auto-backup Drive mỗi 5 phút)

In [ ]:
# Cell 6: Hybrid Retrieve với background backup thread
# → retrieval.db được tự động copy lên Drive MỖI 5 PHÚT
# → Nếu Colab bị ngắt, DB vẫn an toàn trên Drive
import os, shutil, sqlite3, subprocess, threading, time

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
DB_LOCAL  = 'artifacts/cache/retrieval_test.db'
DRIVE_DB  = f'{DRIVE_DIR}/retrieval_hybrid_test.db'
BACKUP_INTERVAL = 300  # giây (5 phút)

# Restore DB từ Drive nếu chạy dở
if os.path.exists(DRIVE_DB) and not os.path.exists(DB_LOCAL):
    shutil.copy2(DRIVE_DB, DB_LOCAL)
    try:
        n = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
        print(f'📦 Restored retrieval.db từ Drive ({n}/2000 câu)')
    except:
        os.remove(DB_LOCAL)
        print('⚠️  DB cũ bị lỗi, bắt đầu lại')

done = 0
if os.path.exists(DB_LOCAL):
    try:
        done = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
    except: done = 0

if done >= 2000:
    print(f'✅ Đã retrieve đủ 2000/2000 câu — Bỏ qua!')
else:
    print(f'🔄 Hybrid Retrieve: còn {2000-done} câu ({done} đã có)...')
    print(f'☁️  Auto-backup Drive mỗi {BACKUP_INTERVAL//60} phút')

    # ── Background backup thread ─────────────────────────────────────
    stop_backup = threading.Event()

    def _backup_loop():
        count = 0
        while not stop_backup.is_set():
            stop_backup.wait(BACKUP_INTERVAL)
            if stop_backup.is_set(): break
            if os.path.exists(DB_LOCAL):
                try:
                    shutil.copy2(DB_LOCAL, DRIVE_DB)
                    n = sqlite3.connect(DB_LOCAL).execute(
                        'SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
                    count += 1
                    print(f'   ☁️  [Auto-backup #{count}] {n}/2000 câu → Drive')
                except Exception as e:
                    print(f'   ⚠️  Backup failed: {e}')

    backup_thread = threading.Thread(target=_backup_loop, daemon=True)
    backup_thread.start()
    # ─────────────────────────────────────────────────────────────────

    try:
        subprocess.run(
            ['python', 'run.py', 'retrieve',
             '--questions', 'data/R2AIStage1DATA.json',
             '--device', 'cuda'],
            capture_output=False
        )
    finally:
        stop_backup.set()  # Dừng backup thread

    # Backup lần cuối sau khi retrieve xong
    if os.path.exists(DB_LOCAL):
        shutil.copy2(DB_LOCAL, DRIVE_DB)
        n = sqlite3.connect(DB_LOCAL).execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
        print(f'\n☁️  Final backup: retrieval_hybrid.db ({n}/2000 câu)')

print('\n✅ Cell 6 Done!')

## Cell 7 — BGE-Reranker + Retune (auto-resume nếu bị ngắt)

In [ ]:
# Cell 7: Reranker + Retune
# → Mỗi 50 câu: lưu checkpoint lên Drive
# → Nếu bị ngắt giữa chừng: chạy lại cell này, nó sẽ TỰ RESUME
import os, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'

# Kiểm tra có checkpoint dở không
ckpt = f'{DRIVE_DIR}/results_reranked_checkpoint_test.json'
if os.path.exists(ckpt):
    import json
    done_count = len(json.loads(open(ckpt, encoding='utf-8').read()))
    print(f'🔄 Tìm thấy checkpoint: {done_count}/2000 câu đã xong → RESUME!')
else:
    print('🆕 Bắt đầu từ đầu (không có checkpoint)')

    print('   HIGH_CONF=0.68 | SAFE=0.58 | MAX_ART=2')
print('   Checkpoint mỗi 50 câu → Drive tự động')

subprocess.run(
    ['python', 'rerank_retune.py',
     '--high-conf', '0.68',
     '--max-art', '2',
     '--device', 'cuda', '--batch-size', '64',
     '--checkpoint-every', '50'],
    capture_output=False
)

out_zip = 'artifacts/output/submission_reranked_test.zip'
if os.path.exists(out_zip):
    sz = os.path.getsize(out_zip) / 1024 / 1024
    print(f'\n🏆 submission_reranked_test.zip: {sz:.1f}MB')
    print(f'☁️  Đã save: {DRIVE_DIR}/submission_reranked_test.zip')
    print('\n📥 TẢI VỀ VÀ NỘP KAGGLE!')
else:
    print('⚠️  Chưa xong! Chạy lại cell này để resume.')
print('\n✅ Cell 7 Done!')

## Cell 8 — (Tùy chọn) Thử threshold khác

In [ ]:
import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'

# ── Chỉnh số này ──
HIGH_CONF = 0.68
SAFE      = 0.58
MAX_ART   = 2
# ──────────────────

print(f'🔧 HIGH_CONF={HIGH_CONF}, SAFE={SAFE}, MAX_ART={MAX_ART}')
subprocess.run(
    ['python', 'rerank_retune.py',
     '--high-conf', str(HIGH_CONF), '--safe', str(SAFE),
     '--min-art', '0', '--max-art', str(MAX_ART),
     '--device', 'cuda', '--batch-size', '64', '--reset'],
    capture_output=False
)
src = 'artifacts/output/submission_reranked.zip'
tag = f'{HIGH_CONF}_{MAX_ART}_{SAFE}'.replace('.', '')
if os.path.exists(src):
    shutil.copy2(src, f'{DRIVE_DIR}/submission_{tag}.zip')
    print(f'✅ submission_{tag}.zip saved to Drive')